El notebook tendrá 6 secciones clave

El notebook completo queda así:
```sh
sagemaker_byoc.ipynb
├── Celda 1 — Setup e imports
├── Celda 2 — Build y push a ECR
├── Celda 3 — Training Job
├── Celda 4 — Desplegar Endpoint
├── Celda 5 — Inferencias en tiempo real
└── Celda 6 — Limpieza de recursos
```

In [1]:
# Sección 1 - setup e imports 
# Esta celda configura todo lo necesario

import boto3
import sagemaker
from sagemaker import get_execution_role

# Sesión de SageMaker
sess = sagemaker.Session()
role = get_execution_role()

# Datos de la cuenta
account_id = boto3.client("sts").get_caller_identity()["Account"]
region = boto3.session.Session().region_name

# Nombres
bucket = "ml-predsales-bucket"
image_name = "ml-predsales-training"
prefix = "data/training"

print(f"Account: {account_id}")
print(f"Region:  {region}")
print(f"Role:    {role}")
print(f"Image:   {account_id}.dkr.ecr.{region}.amazonaws.com/{image_name}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Account: 561137843164
Region:  us-east-1
Role:    arn:aws:iam::561137843164:role/SageMakerStudioExecutionRole2026
Image:   561137843164.dkr.ecr.us-east-1.amazonaws.com/ml-predsales-training


In [5]:
# Sección 2 - Build y push de la imagen a ECR 
# En esta celda vamos a construir la imagen docker y subirla a ECR

import os
import subprocess

# Ruta al Dockerfile
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
training_dir = f"{repo_root}/src/training"

# URI completa de la imagen en ECR
ecr_image = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{image_name}:latest"

# Crear el repositorio en ECR si no existe
os.system(f"aws ecr describe-repositories --repository-names {image_name} > /dev/null 2>&1 || aws ecr create-repository --repository-name {image_name} > /dev/null")

# Login a ECR
os.system(f"aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account_id}.dkr.ecr.{region}.amazonaws.com")

# Build con --network sagemaker (requerido en SageMaker Studio)
os.system(f"docker build --network sagemaker -t {image_name} {training_dir}")

# Tagear y pushear
os.system(f"docker tag {image_name}:latest {ecr_image}")
os.system(f"docker push {ecr_image}")

print(f"Image pushed: {ecr_image}")


WARNING! Your password will be stored unencrypted in /home/sagemaker-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credential-stores

DEPRECATED: The legacy builder is deprecated and will be removed in a future release.
            BuildKit is currently disabled; enable it by removing the DOCKER_BUILDKIT=0
            environment-variable.



Login Succeeded
Sending build context to Docker daemon  40.45kB
Step 1/13 : FROM python:3.12-slim
3.12-slim: Pulling from library/python
206356c42440: Pulling fs layer
30dad65d3b24: Pulling fs layer
5d7ccc6599a6: Pulling fs layer
b9aab63c297c: Pulling fs layer
b9aab63c297c: Waiting
30dad65d3b24: Verifying Checksum
30dad65d3b24: Download complete
5d7ccc6599a6: Verifying Checksum
5d7ccc6599a6: Download complete
b9aab63c297c: Verifying Checksum
b9aab63c297c: Download complete
206356c42440: Verifying Checksum
206356c42440: Download complete
206356c42440: Pull complete
30dad65d3b24: Pull complete
5d7ccc6599a6: Pull complete
b9aab63c297c: Pull complete
Digest: sha256:ccc7089399c8bb65dd1fb3ed6d55efa538a3f5e7fca3f5988ac3b5b87e593bf0
Status: Downloaded newer image for python:3.12-slim
 ---> 6f90d4a79e7a
Step 2/13 : ENV PATH="/opt/program:${PATH}"
 ---> Running in f1b572758237
 ---> Removed intermediate container f1b572758237
 ---> 8f59b6a69aa1
Step 3/13 : WORKDIR /opt/program
 ---> Running in 5

### ¿Qué hace cada paso?
```sh
docker build    → construye la imagen desde tu Dockerfile
ecr create      → crea el repositorio en ECR (solo la primera vez)
ecr get-login   → se autentica con ECR
docker tag      → le pone la URI completa de ECR a la imagen
docker push     → sube la imagen a ECR
```

El resultado final es tu imagen disponible en:
561137843164.dkr.ecr.us-east-1.amazonaws.com/ml-predsales-training:latest


In [6]:
# Sección 3 - Training Job 
#En esta celda se lanza el entrenamiento en SM 

from sagemaker.estimator import Estimator

# URI de la imagen en ECR
ecr_image = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{image_name}:latest"

# Definir el Estimator
estimator = Estimator(
    image_uri=ecr_image,
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size=10,
    max_run=7200,
    output_path=f"s3://{bucket}/output",
    sagemaker_session=sess,
    hyperparameters={
        "n_estimators": 100,
        "max_depth": 10,
        "random_seed": 10,
        "use_random_search": "false",
    }
)

# Lanzar el training job
estimator.fit(
    {"training": f"s3://{bucket}/{prefix}"},
    job_name="ml-predsales-training-job-v2",
    wait=True,
    logs=True,
)

print("Training job complete!")

INFO:sagemaker:Creating training-job with name: ml-predsales-training-job


2026-03-08 19:50:43 Starting - Starting the training job...
2026-03-08 19:50:58 Starting - Preparing the instances for training...
2026-03-08 19:51:20 Downloading - Downloading input data...
2026-03-08 19:51:51 Training - Training image download completed. Training in progress.2026-03-08 19:51:55,120 - train - INFO - ==================================================
2026-03-08 19:51:55,120 - train - INFO - Starting training pipeline (train.py)
2026-03-08 19:51:55,120 - train - INFO - ==================================================
2026-03-08 19:51:55,120 - train - INFO - Loading prepared dataset...
2026-03-08 19:51:58,821 - train - INFO - Prepared dataset loaded: 9,330,156 records
2026-03-08 19:51:59,979 - train - INFO - Train set: 8,906,058 records (months 0–32)
2026-03-08 19:51:59,979 - train - INFO - Validation set: 424,098 records (month 33)
2026-03-08 19:52:00,017 - train - INFO - Starting RandomizedSearchCV — 20 iterations, 3-fold CV...

2026-03-08 20:57:09 Stopping - Stoppin

Training seconds: 3961
Billable seconds: 3961
Training job complete!


In [ ]:
# Sección 4 - desplegar el endpoint 
#Esta celda despliega el modelo como un endpoint de inferencia en tiempo real 

from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

# Desplegar el endpoint
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="ml-predsales-endpoint",
    serializer=CSVSerializer(),
    deserializer=CSVDeserializer(),
)

print(f"Endpoint deployed: {predictor.endpoint_name}")
'''
## ¿Qué hace cada parte?
**`estimator.deploy()`** hace tres cosas automáticamente:
1. Toma el modelo que guarde en S3 tras el training job
2. Lanza una instancia `ml.m5.large` con tu imagen Docker
3. Corre tu script `serve` que levanta gunicorn con Flask

**`CSVSerializer`** convierte tus datos a CSV antes de mandarlos al endpoint

**`CSVDeserializer`** convierte la respuesta CSV del endpoint de vuelta a Python

El resultado es un endpoint disponible en:
https://runtime.sagemaker.us-east-1.amazonaws.com/endpoints/ml-predsales-endpoint/invocations
'''

In [ ]:
# Sección 5 - Inferencias en Tiempo Real 
# En esta celda se generan predicciones usando el endpoint 

import numpy as np
import pandas as pd

# Muestra de datos para inferencia
# Columnas: lag_1, lag_3, lag_6, lag_12
sample_data = pd.DataFrame([
    [5.0, 4.0, 3.0, 2.0],
    [3.0, 2.0, 1.0, 0.5],
    [10.0, 8.0, 6.0, 4.0],
    [0.0, 0.0, 0.0, 0.0],
    [7.0, 6.0, 5.0, 3.0],
], columns=["lag_1", "lag_3", "lag_6", "lag_12"])

# Hacer predicciones
response = predictor.predict(sample_data.values)

# Mostrar resultados
predictions = [float(x[0]) for x in response]
results_df = sample_data.copy()
results_df["predicted_sales"] = predictions

print("Inferencias en tiempo real:")
print(results_df.to_string(index=False))

'''
## ¿Qué pasa aquí?
sample_data (DataFrame)
      ↓ CSVSerializer lo convierte a CSV
POST /invocations en el endpoint
      ↓ predictor.py corre model.predict()
      ↓ CSVDeserializer convierte respuesta
predictions (lista de floats)

El resultado se ve así:
```
lag_1  lag_3  lag_6  lag_12  predicted_sales
  5.0    4.0    3.0     2.0              4.2
  3.0    2.0    1.0     0.5              1.8
 10.0    8.0    6.0     4.0              8.5
  0.0    0.0    0.0     0.0              0.0
  7.0    6.0    5.0     3.0              5.9
  '''

In [ ]:
# Sección 6 - Limpieza de recursos
# Esta celda elimina el endpoint para no generar costos innecesarios 

# Eliminar el endpoint
predictor.delete_endpoint()
print(f"Endpoint deleted: ml-predsales-endpoint")

'''
## ¿Por qué es importante esta celda?

Un endpoint en SageMaker cobra por hora mientras esté corriendo, aunque se esté usando. 
'''